# 01 — Basic LLM Calls: The Starting Point

Notebook 00 ended with a promise: paste Arjun's bookshop story into the prompt ourselves, by hand, and watch it actually work. Today we do exactly that — and then we keep pushing on it until it breaks, because that break is what the rest of this series exists to fix.

Here's the full story we'll use throughout this notebook.


> **The Portal Bookshop**
>
> Arjun had walked past the same bookshop in Hyderabad's Abids market a hundred times without a second glance — dusty windows, a hand-painted sign, nothing special. But tonight the door was open just a crack, spilling warm yellow light onto the wet street after the evening rain. He told himself he was only stepping in to get out of the drizzle.
>
> Inside, towers of yellowed paperbacks leaned against each other like old friends. The shopkeeper was nowhere to be seen. At the back, past a shelf of forgotten atlases, stood a second doorway that hadn't been there on his last visit — its outline shimmering faintly, like heat rising off summer asphalt.
>
> Arjun told himself it was a trick of the light. He stepped closer anyway.
>
> The moment he crossed the threshold, the smell of old paper vanished, replaced by salt air and the low rush of waves. He was standing barefoot on cool sand, under a sky holding two moons instead of one. Behind him, the bookshop door was simply gone — no wall, no street, no Hyderabad. Just the dunes, the strange twin light, and a set of footprints in the sand that weren't his own, leading toward the water.
>
> Somewhere past the first dune, someone was singing in a language he didn't recognize, but somehow understood every word of.
>
> Arjun took a breath, and followed the footprints.


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Same model as notebook 00: Groq's free-tier replacement for the
# deprecated llama-3.1-8b-instant.
MODEL = "openai/gpt-oss-20b"

portal_bookshop_story = """
Arjun had walked past the same bookshop in Hyderabad's Abids market a
hundred times without a second glance -- dusty windows, a hand-painted
sign, nothing special. But tonight the door was open just a crack,
spilling warm yellow light onto the wet street after the evening rain.
He told himself he was only stepping in to get out of the drizzle.

Inside, towers of yellowed paperbacks leaned against each other like
old friends. The shopkeeper was nowhere to be seen. At the back, past
a shelf of forgotten atlases, stood a second doorway that hadn't been
there on his last visit -- its outline shimmering faintly, like heat
rising off summer asphalt.

Arjun told himself it was a trick of the light. He stepped closer
anyway.

The moment he crossed the threshold, the smell of old paper vanished,
replaced by salt air and the low rush of waves. He was standing
barefoot on cool sand, under a sky holding two moons instead of one.
Behind him, the bookshop door was simply gone -- no wall, no street,
no Hyderabad. Just the dunes, the strange twin light, and a set of
footprints in the sand that weren't his own, leading toward the water.

Somewhere past the first dune, someone was singing in a language he
didn't recognize, but somehow understood every word of.

Arjun took a breath, and followed the footprints.
"""


## Take 1 — ask without giving it anything

Same test as notebook 00, just to confirm the model still doesn't know this story on its own.


In [ ]:
def ask_llm(question: str) -> str:
    """Bare call -- no context, nothing but the model's own memory."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


print("LLM Response:", ask_llm("What happens to Arjun in The Portal Bookshop story?"))


Still a guess — exactly like Kalki 2898 AD in notebook 00. The model has never read this story, so it's making something up that merely sounds plausible. Time to actually fix that.


## What's actually inside an LLM call

Before we inject the story, it's worth seeing what we've been hiding inside `ask_llm()` — every call to a chat model is really a small list of messages, not just one string. Each message has a **role**:

- **`system`** — persistent background instructions, set once, applies to the whole conversation
- **`user`** — the actual question for this turn
- **`assistant`** — the model's reply (this is what comes back in the response)

Two more things worth knowing before we go further:

- **Tokens** — same rule of thumb as notebook 00: 1 token ≈ 4 letters. The model reads your whole messages list (system + user + prior turns) as tokens, and there's a hard cap on how many fit.
- **Temperature** — how much the model is allowed to "wing it." `0` is as deterministic as it gets — same input, (almost) same output every time. `1` lets it wander and get more creative. For factual Q&A like ours, low temperature (`0`–`0.2`) is what you want; a creative writing task might use `0.7`+.

Let's look at the raw shape of a request and response, once, so `ask_llm()` stops feeling like a black box.


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What happens to Arjun in the bookshop story?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.2, reasoning_effort="low")

print("Full response object:\n", response)
print("\nJust the text we actually want:\n", response.choices[0].message.content)


Everything we care about — the model's reply — lives at `response.choices[0].message.content`. Everything else (token counts, model name, timing) is bookkeeping. Now let's actually put the story to use.


## Take 2 — hand it the story, right in the question

The simplest possible fix: paste the story into the user message, right alongside the question.


In [ ]:
def ask_llm_with_context(question: str, context: str) -> str:
    """Inject context directly into the user message."""
    prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{question}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


answer = ask_llm_with_context(
    "What happens to Arjun in the bookshop story?",
    portal_bookshop_story,
)
print("LLM Response:", answer)


It works. The model now has the story in front of it, so it's answering from the text instead of guessing. Notice what we did, though — we glued the story and the question together into one big block of text and called it the "user message." That's a bit sloppy: the story isn't really part of *this turn's question*, it's background the model should already have before the question even shows up.


## Take 3 — hand it the story as a standing instruction

This is what the **system** role is actually for: background that should apply no matter what the user asks. Let's move the story there, and keep the user message as just the question.


In [ ]:
def ask_llm_with_system_context(question: str, context: str) -> str:
    """Context lives in the system message; the user message is just the question."""
    system_prompt = f"""You are StoryVerse AI. Answer questions using ONLY the context below.

Context:
{context}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=0.2,
        reasoning_effort="low",
    )
    return response.choices[0].message.content


answer = ask_llm_with_system_context(
    "What happens to Arjun in the bookshop story?",
    portal_bookshop_story,
)
print("LLM Response:", answer)


Same correct answer, cleaner separation: the system message holds *what the model should know*, the user message holds *what's being asked right now*. This matters once a conversation has more than one question — the story only needs to be stated once, not repeated into every user message.

But notice something we haven't fixed yet: the story is still typed straight into this notebook as a Python string. What happens when we have 50 stories, not 1?


## The wall we hit again

Let's find out. Here are four more short stories from the StoryVerse universe, alongside Arjun's.


In [ ]:
stories = {
    "interstellar": """
Cooper, a former NASA pilot turned farmer, discovers a hidden NASA
facility and is recruited for a mission to save humanity from a dying
Earth. Traveling through a wormhole near Saturn, his crew searches
distant planets for a new home. Time dilation near a black hole costs
Cooper decades with his daughter Murph. He eventually falls into the
black hole, ending up in a tesseract built by future humans, where he
sends her the gravitational data needed to save Earth.
""",
    "harry_potter": """
Harry Potter, an orphan raised by unkind relatives, learns on his
eleventh birthday that he is a wizard and is enrolled at Hogwarts
School of Witchcraft and Wizardry. He makes friends with Ron Weasley
and Hermione Granger, learns his parents were killed by the dark
wizard Voldemort, and discovers that Voldemort is after the Sorcerer's
Stone, hidden in the school. Harry and his friends stop Voldemort from
obtaining it.
""",
    "baahubali": """
Shivudu grows up in a remote village, unaware he is the son of a
murdered king. He falls in love with a warrior, Avanthika, and follows
her to the kingdom of Mahishmati, where he learns of his true heritage
and the betrayal of his uncle Bhallaladeva. He takes up his father's
mace, rallies the oppressed people, and moves to reclaim his rightful
throne.
""",
    "dark_knight": """
Batman, District Attorney Harvey Dent, and Commissioner Gordon attempt
to dismantle organized crime in Gotham City. The Joker, an anarchic
criminal mastermind, escalates the chaos, manipulating Harvey Dent
into becoming the vengeful Two-Face after a personal tragedy. Batman
takes the blame for Dent's crimes to protect Gotham's faith in its
white knight, becoming a fugitive in the process.
""",
    "portal_bookshop": portal_bookshop_story,
}

for name, text in stories.items():
    print(f"{name:20} {len(text):>5} chars")


In [ ]:
combined_context = "\n\n".join(stories.values())
print(f"5 stories combined: {len(combined_context):,} chars (~{len(combined_context)//4:,} tokens)")

answer = ask_llm_with_system_context(
    "What happens to Arjun in the bookshop story?",
    combined_context,
)
print("\nLLM Response:", answer)


Still works — 5 stories is nothing. But watch what happens as StoryVerse AI grows from a hobby project into something with a real catalog.


In [ ]:
avg_story_chars = len(combined_context) // len(stories)

for num_stories in [5, 50, 500, 5000]:
    total_chars = num_stories * avg_story_chars
    est_tokens = total_chars // 4
    print(f"{num_stories:>5} stories -> {total_chars:>9,} chars -> ~{est_tokens:>8,} tokens")


Same wall as notebook 00's context window, hit from a different direction: at 5 stories we're fine, and by 500 we've already blown past what most context windows hold — for a question that only ever needed *one* of those stories. We're paying to send Batman trivia to answer a question about a Hyderabad bookshop.


## The real problem: our code is doing too many jobs

There's a simple engineering idea called the **Single Responsibility Principle (SRP)**: a function — or in this case, a notebook — should do one thing. Look at what ours is doing right now, all tangled together:

1. **Storing knowledge** — the story strings, hardcoded in a Python dict
2. **Formatting prompts** — building the system message
3. **Calling the model** — the actual API request
4. **Returning answers** — handing back the text

That's four jobs in one place. If a story's plot needs a correction, you edit this notebook's source code. If you want to add story #6, you edit this notebook's source code. Knowledge and logic are stuck together, and that's exactly backwards — the *stories* should be able to change without anyone touching the *code*.

**Next notebook:** we pull the stories out into their own files, load them from disk instead of hardcoding them, and see the same pipeline through both plain Python and LangChain — so you can see LangChain is doing the same thing we just did by hand, just with reusable pieces.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| System message | Standing background info, set once, applies to the whole conversation |
| User message | The specific question being asked this turn |
| Assistant message | The model's reply |
| Temperature | How much freedom the model has to deviate from the safest answer |
| Single Responsibility Principle (SRP) | One function, one job — don't tangle storage, formatting, and calling together |

We went from a bare, guessing model, to correctly answering from injected context, to hitting the same scaling wall from notebook 00 — this time because *our own code* hardcodes the knowledge instead of loading it.

**Exercises:**

- Change `temperature` to `1.0` in `ask_llm_with_system_context` and ask the same question a few times — notice the answer drift.
- Call `ask_llm_with_system_context` with an *empty* context string and ask a specific plot detail — the hallucination comes right back.
- Add a 6th story of your own to `stories`, and ask a question that only your new story can answer.
